## 1. Environment Preparation

### 1.2 Libraries

In [ ]:
import time
import json
import pandas as pd
from tqdm import tqdm
import paho.mqtt.client as mqtt
from datetime import datetime
from openelectricity import OEClient, MarketMetric, DataMetric

### 1.3 Global Variables

In [ ]:
# CSV save path
CSV_NAME = "a2_data.csv"

# Load API key
API_KEY = ""
BASE_URL = "https://api.openelectricity.org.au/v4"
HEADERS = {"Authorization": f"Bearer {API_KEY}", "Accept-Encoding": "gzip, deflate"}
DATE_START = datetime(2026, 5, 1, 0, 0, 0)
DATE_END = datetime(2026, 5, 8, 0, 0, 0)
BATCH_SIZE = 30
NETWORK = "NEM"
INTERVAL = "5m"

# MQTT broker configuration
MQTT_BROKER = "broker.hivemq.com"
MQTT_PORT = 1883
MQTT_TOPIC = "comp5339/a2/group80/facility_power_emissions"

## 2. Data Retrieval

### 2.1 Helper Functions

In [5]:
# Helper function for retrieve facility
def retrieve_facility():
    response = client.get_facilities(network_id=NETWORK)
    rows = []
    for facility in response.data:
        if facility.location is None:
            continue
        for unit in facility.units:
            rows.append({
                "unit_code": unit.code,
                "facility_code": facility.code,
                "facility_name": facility.name,
                "network_region": facility.network_region,
                "fueltech_id": unit.fueltech_id,
                "latitude": facility.location.lat,
                "longitude": facility.location.lng
            })
    return pd.DataFrame(rows)

In [6]:
# Helper function for retrieve facility power and emission
def retrieve_power_emission(df):
    rows_power = []
    rows_emission = []
    facility_code = df['facility_code'].unique().tolist()
    for i in range(0, len(facility_code), BATCH_SIZE):
        response = client.get_facility_data(
            network_code=NETWORK,
            metrics=[DataMetric.POWER, DataMetric.EMISSIONS],
            date_start=DATE_START,
            date_end=DATE_END,
            facility_code=facility_code[i:i + BATCH_SIZE],
            interval=INTERVAL
        )
        for result in response.data[0].results:
            for data in result.data:
                rows_power.append({
                    "unit_code": result.columns.unit_code,
                    "time": data.root[0],
                    "power (" + response.data[0].unit + ")": data.root[1],
                })
        for result in response.data[1].results:
            for data in result.data:
                rows_emission.append({
                    "unit_code": result.columns.unit_code,
                    "time": data.root[0],
                    "emission (" + response.data[1].unit + ")": data.root[1],
                })
    return pd.DataFrame(rows_power), pd.DataFrame(rows_emission)

In [7]:
# Helper function for retrieve market price and demand
def retrieve_price_demand():
    rows_price = []
    rows_demand = []
    response = client.get_market(
        network_code=NETWORK,
        metrics=[MarketMetric.PRICE, MarketMetric.DEMAND],
        date_start=DATE_START,
        date_end=DATE_END,
        interval=INTERVAL,
        primary_grouping='network_region'
    )
    for result in response.data[0].results:
        for data in result.data:
            rows_price.append({
                "network_region": result.name.split("_")[1],
                "time": data.root[0],
                "region_price (per " + response.data[0].unit[2:] + ")": data.root[1],
            })
    for result in response.data[1].results:
        for data in result.data:
            rows_demand.append({
                "network_region": result.name.split("_")[1],
                "time": data.root[0],
                "region_demand (" + response.data[1].unit + ")": data.root[1],
            })
    return pd.DataFrame(rows_price), pd.DataFrame(rows_demand)

### 2.2 Dowload Data

In [8]:
# Download all data
client = OEClient(api_key=API_KEY)
df_facility = retrieve_facility()
df_power, df_emission = retrieve_power_emission(df_facility)
df_price, df_demand = retrieve_price_demand()

In [9]:
# Display facility data
df_facility.head(3)

,unit_code,facility_code,facility_name,network_region,fueltech_id,latitude,longitude
0,ADPBA1L,ADP,Adelaide Desalination,SA1,battery_charging,-35.100751,138.483357
1,ADPBA1G,ADP,Adelaide Desalination,SA1,battery_discharging,-35.100751,138.483357
2,ADPBA1,ADP,Adelaide Desalination,SA1,battery,-35.100751,138.483357


In [10]:
# Display power data
df_power.head(3)

,unit_code,time,power (MW)
0,ADPBA1,2026-05-01 00:00:00+10:00,-0.011
1,ADPBA1,2026-05-01 00:05:00+10:00,-0.021
2,ADPBA1,2026-05-01 00:10:00+10:00,0.106


In [11]:
# Display emission data
df_emission.head(3)

,unit_code,time,emission (t)
0,ADPBA1,2026-05-01 00:00:00+10:00,0.0
1,ADPBA1,2026-05-01 00:05:00+10:00,0.0
2,ADPBA1,2026-05-01 00:10:00+10:00,0.0


In [12]:
# Display price data
df_price.head(3)

,network_region,time,region_price (per MWh)
0,NSW1,2026-05-01 00:00:00+10:00,57.51
1,NSW1,2026-05-01 00:05:00+10:00,57.51
2,NSW1,2026-05-01 00:10:00+10:00,65.89


In [13]:
# Display demand data
df_demand.head(3)

,network_region,time,region_demand (MW)
0,NSW1,2026-05-01 00:00:00+10:00,7332.50
1,NSW1,2026-05-01 00:05:00+10:00,7323.78
2,NSW1,2026-05-01 00:10:00+10:00,7441.25


In [16]:
client.close()

### 2.3 Task 1 Summary

In [14]:
print("=" * 10 + " Task 1 Summary " + "=" * 10)
print(f"{'Time Start:':<25} {DATE_START}")
print(f"{'Time End:':<25} {DATE_END}")
print(f"{'Facilities:':<25} {len(df_facility)}")
print(f"{'Power records:':<25} {len(df_power)}")
print(f"{'Emissions records:':<25} {len(df_emission)}")
print(f"{'Price records:':<25} {len(df_price)}")
print(f"{'Demand records:':<25} {len(df_demand)}")

========== Task 1 Summary ==========
Time Start:               2026-05-01 00:00:00
Time End:                 2026-05-08 00:00:00
Facilities:               944
Power records:            1074070
Emissions records:        1074070
Price records:            12096
Demand records:           12096


## 3. Data Integration and Materialisation/Caching

### 3.1 Heiper Functions

In [15]:
# Merge facility, power and emissions
def merge_facility_power_emissions(df_power, df_emission, df_facility):
  df_merged = (
    pd.merge(
      df_power,
      df_emission,
      on=["time", "unit_code"],
      how="outer"
    )
    .merge(
      df_facility,
      on="unit_code",
      how="inner"
    )
  )
  return df_merged

In [17]:
# Aggregate data
def aggregate_by_facility(df):
  return (
    df.groupby(
      ["time", "facility_code", "facility_name", "network_region", "fueltech_id", "latitude", "longitude"],
      dropna=False
    )[df.filter(regex=r'^power\s*\(.*\)$').columns.tolist() +
      df.filter(regex=r'^emission\s*\(.*\)$').columns.tolist()]
    .sum()
    .reset_index()
    .sort_values(["time", "facility_code"])
    .reset_index(drop=True)
  )

In [18]:
# Merge market price and demand
def merge_market_price_demand(df, df_price, df_demand):
  df_merged = (
    pd.merge(
      df,
      df_price,
      on=["time", "network_region"],
      how="left"
    )
    .merge(
      df_demand,
      on=["time", "network_region"],
      how="left"
    )
  )
  return df_merged

### 3.2 Integrate Data

In [19]:
# Merge and aggregate data
df_merged = merge_facility_power_emissions(df_power, df_emission, df_facility)
df_merged = aggregate_by_facility(df_merged)
df_final = merge_market_price_demand(df_merged, df_price, df_demand)
df_final.head(10)

,time,facility_code,facility_name,network_region,fueltech_id,latitude,longitude,power (MW),emission (t),region_price (per MWh),region_demand (MW)
0,2026-05-01 00:00:00+10:00,ADP,Adelaide Desalination,SA1,battery,-35.100751,138.483357,-0.0110,0.0000,-2.01,1507.69
1,2026-05-01 00:00:00+10:00,ADP,Adelaide Desalination,SA1,battery_charging,-35.100751,138.483357,0.0110,0.0000,-2.01,1507.69
2,2026-05-01 00:00:00+10:00,ADP,Adelaide Desalination,SA1,solar_utility,-35.100751,138.483357,0.0000,0.0000,-2.01,1507.69
3,2026-05-01 00:00:00+10:00,AGLHAL,Hallett,SA1,gas_ocgt,-33.349528,138.751607,0.0000,0.0000,-2.01,1507.69
4,2026-05-01 00:00:00+10:00,AGLSOM,Somerton,VIC1,gas_ocgt,-37.631692,144.952802,0.0000,0.0000,4.55,4568.84
5,2026-05-01 00:00:00+10:00,ALDGASF,Aldoga,QLD1,solar_utility,-23.839544,151.084900,0.0000,0.0000,64.65,6165.20
6,2026-05-01 00:00:00+10:00,ANGASTON,Angaston,SA1,distillate,-34.503389,139.024580,0.0000,0.0000,-2.01,1507.69
7,2026-05-01 00:00:00+10:00,ARWF,Ararat,VIC1,wind,-37.263393,143.082116,99.5000,0.0000,4.55,4568.84
8,2026-05-01 00:00:00+10:00,AVLSF,Avonlie,NSW1,solar_utility,-34.913826,146.590545,0.0000,0.0000,57.51,7332.50
9,2026-05-01 00:00:00+10:00,B2PS,Braemar 2,QLD1,gas_ocgt,-27.112874,150.905442,0.0604,0.0029,64.65,6165.20


In [20]:
df_final.to_csv(CSV_NAME, index=False)

### 3.3 Task 2 Summery

In [21]:
print("=" * 10 + " Task 2 Summary " + "=" * 10)
print(f"{'Time Start:':<25} {DATE_START}")
print(f"{'Time End:':<25} {DATE_END}")
print(f"{'Merged records:':<25} {len(df_final)}")
print(f"{'Merged records:':<25} {len(df_final)}")

========== Task 2 Summary ==========
Time Start:               2026-05-01 00:00:00
Time End:                 2026-05-08 00:00:00
Merged records:           868438
Merged records:           868438


## 4. Data Publishing via MQTT

In [22]:
# MQTT Publisher
def publish_facility_stream(df):
    df = df.sort_values(["time", "facility_code"])
    print(f"Publishing {len(df)} records...")
    client = mqtt.Client(callback_api_version=mqtt.CallbackAPIVersion.VERSION2)
    client.connect(MQTT_BROKER, MQTT_PORT, 60)
    client.loop_start()
    for _, row in tqdm(df.iterrows(), total=len(df), desc="MQTT Publishing"):
        msg = {
            "time": str(row["time"]),
            "facility_code": row["facility_code"],
            "facility_name": row["facility_name"],
            "network_region": row["network_region"],
            "fueltech_id": row.get("fueltech_id"),
            "latitude": row.get("latitude"),
            "longitude": row.get("longitude"),
            "power_mw": row.get("power (MW)"),
            "emission_t": row.get("emission (t)"),
            "region_price_per_mwh": row.get("region_price (per MWh)"),
            "region_demand_mw": row.get("region_demand (MW)")
        }
        payload = json.dumps(msg)
        client.publish(MQTT_TOPIC, payload)
        time.sleep(0.1)
    client.loop_stop()
    client.disconnect()
    print("Published All")

In [23]:
df = pd.read_csv("a2_data.csv")
try:
    publish_facility_stream(df)
except KeyboardInterrupt:
    print("\nInterrupted by keyboard")

Publishing 868438 records...


MQTT Publishing:   0%|          | 320/868438 [00:33<24:54:15,  9.68it/s]



Interrupted by keyboard


## 5. Continuous Execution

In [23]:
# Simulate an unbounded data
def simulate_unbounded_data_stream():
  while True:
    print("=== Starting new round ===")

    # Data Retrieval
    print("Retrieving data ...")
    client = OEClient(api_key=API_KEY)
    df_facility = retrieve_facility()
    df_power, df_emission = retrieve_power_emission(df_facility)
    df_price, df_demand = retrieve_price_demand()
    client.close()

    # Data Preprocessing
    print("Processing data ...")
    df_merged = merge_facility_power_emissions(df_power, df_emission, df_facility)
    df_merged = aggregate_by_facility(df_merged)
    df_merged = merge_market_price_demand(df_merged, df_price, df_demand)

    # Data Publishing
    print("Publishing data ...")
    publish_facility_stream(df_merged)
    print("=== Round complete. Sleeping 60 seconds ===")
    time.sleep(60)

In [24]:
try:
    simulate_unbounded_data_stream()
except KeyboardInterrupt:
    print("\nInterrupted by keyboard")

=== Starting new round ===
Retrieving data ...
Processing data ...
Publishing data ...
Publishing 868438 records...


MQTT Publishing:   4%|▍         | 36734/868438 [1:02:28<23:34:22,  9.80it/s]



Interrupted by keyboard
